# Unlearning Project — Reproducibility and Historical Mega Notebook

**Purpose:** consolidate the work completed before LangChain/LangGraph into one audit-first notebook.

This notebook supports:

1. personal verification of every historical experiment;
2. reconstruction of the project history without mixing incompatible runs;
3. regeneration of metrics from saved raw outputs;
4. reproducible future experiments with controlled configurations;
5. a later transition to Git-based version control.

> **Safe default:** all paid API calls are disabled.

> **Reproducibility note:** a seed controls local randomness, but it cannot by itself guarantee identical outputs from hosted LLMs. This notebook also records data hashes, prompt hashes, model identifiers, SDK versions, timestamps, configurations, and raw responses.

## Recommended workflow

1. Keep this notebook at the project root.
2. Put all original notebooks and outputs into `archive/`.
3. Do not rename or edit historical artifacts initially.
4. Run the manifest and audit sections.
5. Compare regenerated values against the final consolidated report.
6. Add missing exact dates from notebook logs when available.
7. Freeze a verified run bundle.
8. Only then refactor reusable logic into source files and initialize Git.

# 1. Historical project map

Exact experiment dates are used when recoverable. A date described as a **file date** is useful provenance but may not be the original execution date.

### `mini_unlearning_pipeline_api_pymupdf.ipynb` — file date: 2026-05-15
Initial demonstration: PDF extraction, prompting, structured responses, evaluation, and exports.

### `Unlearning_AB_Test_Pipeline_v2.ipynb` — file date: 2026-07-03
Four prompt variants, multiple providers, JSONL resume, token/cost tracking, and budget guards.

### `Unlearning_AB_Test_Pipeline_v3_fixed_api.ipynb` — exact date not recovered
Controlled prompt-parity test on the original 11 positive passages.

### `Unlearning_AB_Test_Pipeline_v4_PROVIDER_SEPARATED_FIXED.ipynb` — exact run date not recovered
Provider-separated benchmark. This stage contains the Claude Sonnet definitions-only 11/11 result.

### `Unlearning_Definitions_Only_Pipeline_Combined_Human_Test_Set.ipynb` — file date: 2026-07-09
Mixed-label definitions-only benchmark on 49 passages.

### `Unlearning_Weekly_Update_Review_Workbook.xlsx` — file date: 2026-07-10
False-negative, document-level, confidence, cost, and review analysis.

### `Unlearning_Local_Context_Definitions_Only_FIXED.ipynb` — file date: 2026-07-17
Definitions-only benchmark using bounded local source context.

### `Unlearning_PostHoc_Cleaned_Metrics.xlsx` — exact date not recovered
Duplicate- and leakage-adjusted strategy comparison.

### `Unlearning_Production_MultiDocument_Annotation_Pipeline_v2_Verified_Extraction.ipynb` — exact date not recovered
Verified extraction, model consensus, reliability, and editable PDF annotation.

### `Unlearning_Project_Final_Results_Report.xlsx` — file date: 2026-07-22
Primary final verification target.

## Comparison rules

- The original 11-row benchmark contained only human-positive passages. It measures **positive recall**, not mixed-class accuracy.
- The 49-row and 42-row benchmarks contain Yes and No labels, so accuracy, precision, recall, specificity, F1, and kappa are meaningful.
- Few-shot evaluation must exclude direct test/example overlap.
- Production documents without human gold labels should be evaluated through agreement, disagreement review, rationales, and adjudication—not accuracy.

# 2. Environment and reproducibility contract

In [1]:
# Optional installation cell. Run once in a fresh environment and restart the kernel.
#
# %pip install -U pandas numpy openpyxl scikit-learn statsmodels krippendorff #     python-dotenv tqdm nbformat openai anthropic google-genai #     pymupdf pdfplumber pyarrow

In [2]:
from __future__ import annotations

from dataclasses import dataclass, asdict, field
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Optional
from collections import Counter
import contextlib
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import sys
import time
import uuid

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score
)

try:
    from statsmodels.stats.inter_rater import fleiss_kappa
except ImportError:
    fleiss_kappa = None

try:
    import krippendorff
except ImportError:
    krippendorff = None

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

In [3]:
@dataclass(frozen=True)
class ProjectConfig:
    project_name: str = "unlearning"
    notebook_version: str = "mega_v1"
    base_seed: int = 20260722
    timezone_name: str = "America/New_York"

    run_api_calls: bool = False
    strict_missing_files: bool = False
    overwrite_outputs: bool = False

    temperature: float = 0.0
    max_output_tokens: int = 2048
    request_sleep_seconds: float = 0.2
    max_estimated_cost_usd: float = 5.0

    text_column: str = "Text Content"
    id_column: str = "Number"
    gold_column: str = "Unlearning"
    document_column: str = "Document"
    context_column: str = "Local Context (±2 source paragraphs)"

    project_root: Path = Path.cwd()

CONFIG = ProjectConfig()

PROJECT_ROOT = CONFIG.project_root
ARCHIVE_DIR = PROJECT_ROOT / "archive"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RUNS_DIR = PROJECT_ROOT / "runs"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [ARCHIVE_DIR, DATA_DIR, OUTPUTS_DIR, RUNS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([asdict(CONFIG)]).T.rename(columns={0: "value"}))

,value
project_name,unlearning
notebook_version,mega_v1
base_seed,20260722
timezone_name,America/New_York
run_api_calls,False
strict_missing_files,False
overwrite_outputs,False
temperature,0.0
max_output_tokens,2048
request_sleep_seconds,0.2


## Seed policy

The seed controls local shuffling, sampling, data splits, bootstrap analysis, and local ML libraries. Hosted LLM output can still drift because model snapshots, routing, safety systems, and provider infrastructure can change.

In [4]:
def set_global_seed(seed: int) -> dict[str, Any]:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    status = {
        "python_random_seed": seed,
        "numpy_seed": seed,
        "pythonhashseed_environment": os.environ["PYTHONHASHSEED"],
        "torch_seeded": False,
        "tensorflow_seeded": False,
    }

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        with contextlib.suppress(Exception):
            torch.use_deterministic_algorithms(True)
        status["torch_seeded"] = True
    except ImportError:
        pass

    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
        status["tensorflow_seeded"] = True
    except ImportError:
        pass

    return status

SEED_STATUS = set_global_seed(CONFIG.base_seed)
SEED_STATUS

{'python_random_seed': 20260722,
 'numpy_seed': 20260722,
 'pythonhashseed_environment': '20260722',
 'torch_seeded': True,
 'tensorflow_seeded': False}

In [5]:
TRACKED_PACKAGES = [
    "pandas", "numpy", "openpyxl", "scikit-learn", "statsmodels",
    "krippendorff", "openai", "anthropic", "google-genai",
    "PyMuPDF", "pdfplumber", "python-dotenv", "tqdm", "nbformat"
]

def package_version(name: str) -> Optional[str]:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

ENVIRONMENT_SNAPSHOT = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "executable": sys.executable,
    "platform": platform.platform(),
    "packages": {name: package_version(name) for name in TRACKED_PACKAGES},
    "seed_status": SEED_STATUS,
}

display(pd.DataFrame(
    [{"package": k, "version": v} for k, v in ENVIRONMENT_SNAPSHOT["packages"].items()]
))

,package,version
0,pandas,2.2.3
1,numpy,2.3.5
2,openpyxl,3.1.5
3,scikit-learn,1.8.0
4,statsmodels,0.14.6
5,krippendorff,None
6,openai,None
7,anthropic,None
8,google-genai,None
9,PyMuPDF,1.26.7


# 3. Historical artifact registry

In [6]:
ARTIFACT_REGISTRY = pd.DataFrame([
    ["demo", "notebook", "mini_unlearning_pipeline_api_pymupdf.ipynb",
     "2026-05-15", "file-library creation date",
     "Initial PDF-to-annotation demonstration", "historical"],
    ["v2", "notebook", "Unlearning_AB_Test_Pipeline_v2.ipynb",
     "2026-07-03", "file-library creation date",
     "Early four-prompt multi-provider pipeline", "historical"],
    ["v3", "notebook", "Unlearning_AB_Test_Pipeline_v3_fixed_api.ipynb",
     None, "not recovered",
     "Controlled 11-row prompt-parity experiment", "historical"],
    ["v4", "notebook", "Unlearning_AB_Test_Pipeline_v4_PROVIDER_SEPARATED_FIXED.ipynb",
     None, "run date not recovered",
     "Provider-separated grid and Claude 11/11 result", "historical"],
    ["v5", "notebook", "Unlearning_Definitions_Only_Pipeline_Combined_Human_Test_Set.ipynb",
     "2026-07-09", "file-library creation date",
     "Five-model 49-row definitions-only benchmark", "current analysis"],
    ["v5_analysis", "workbook", "Unlearning_Weekly_Update_Review_Workbook.xlsx",
     "2026-07-10", "file-library creation date",
     "False-negative and document-level review", "current analysis"],
    ["v6", "notebook", "Unlearning_Local_Context_Definitions_Only_FIXED.ipynb",
     "2026-07-17", "file-library creation date",
     "42-row local-context benchmark", "current analysis"],
    ["cleaned_audit", "workbook", "Unlearning_PostHoc_Cleaned_Metrics.xlsx",
     None, "not recovered",
     "Duplicate- and leakage-adjusted comparison", "post-hoc diagnostic"],
    ["production_v2", "notebook",
     "Unlearning_Production_MultiDocument_Annotation_Pipeline_v2_Verified_Extraction.ipynb",
     None, "not recovered",
     "Verified extraction, consensus, reliability, annotations", "verified production"],
    ["final_report", "workbook", "Unlearning_Project_Final_Results_Report.xlsx",
     "2026-07-22", "file-library creation date",
     "Final consolidated verification target", "final report"],
], columns=[
    "stage", "artifact_type", "filename", "known_date", "date_basis",
    "purpose", "evidence_status"
])

ARTIFACT_REGISTRY["expected_path"] = ARTIFACT_REGISTRY["filename"].map(
    lambda name: str(ARCHIVE_DIR / name)
)
display(ARTIFACT_REGISTRY)

,stage,artifact_type,filename,known_date,date_basis,purpose,evidence_status,expected_path
0,demo,notebook,mini_unlearning_pipeline_api_pymupdf.ipynb,2026-05-15,file-library creation date,Initial PDF-to-annotation demonstration,historical,/mnt/data/archive/mini_unlearning_pipeline_api_pymupdf.ipynb
1,v2,notebook,Unlearning_AB_Test_Pipeline_v2.ipynb,2026-07-03,file-library creation date,Early four-prompt multi-provider pipeline,historical,/mnt/data/archive/Unlearning_AB_Test_Pipeline_v2.ipynb
2,v3,notebook,Unlearning_AB_Test_Pipeline_v3_fixed_api.ipynb,None,not recovered,Controlled 11-row prompt-parity experiment,historical,/mnt/data/archive/Unlearning_AB_Test_Pipeline_v3_fixed_api.ipynb
3,v4,notebook,Unlearning_AB_Test_Pipeline_v4_PROVIDER_SEPARATED_FIXED.ipynb,None,run date not recovered,Provider-separated grid and Claude 11/11 result,historical,/mnt/data/archive/Unlearning_AB_Test_Pipeline_v4_PROVIDER_SEPARATED_FIXED.ipynb
4,v5,notebook,Unlearning_Definitions_Only_Pipeline_Combined_Human_Test_Set.ipynb,2026-07-09,file-library creation date,Five-model 49-row definitions-only benchmark,current analysis,/mnt/data/archive/Unlearning_Definitions_Only_Pipeline_Combined_Human_Test_Set.ipynb
5,v5_analysis,workbook,Unlearning_Weekly_Update_Review_Workbook.xlsx,2026-07-10,file-library creation date,False-negative and document-level review,current analysis,/mnt/data/archive/Unlearning_Weekly_Update_Review_Workbook.xlsx
6,v6,notebook,Unlearning_Local_Context_Definitions_Only_FIXED.ipynb,2026-07-17,file-library creation date,42-row local-context benchmark,current analysis,/mnt/data/archive/Unlearning_Local_Context_Definitions_Only_FIXED.ipynb
7,cleaned_audit,workbook,Unlearning_PostHoc_Cleaned_Metrics.xlsx,None,not recovered,Duplicate- and leakage-adjusted comparison,post-hoc diagnostic,/mnt/data/archive/Unlearning_PostHoc_Cleaned_Metrics.xlsx
8,production_v2,notebook,Unlearning_Production_MultiDocument_Annotation_Pipeline_v2_Verified_Extraction.ipynb,None,not recovered,"Verified extraction, consensus, reliability, annotations",verified production,/mnt/data/archive/Unlearning_Production_MultiDocument_Annotation_Pipeline_v2_Verified_Extraction.ipynb
9,final_report,workbook,Unlearning_Project_Final_Results_Report.xlsx,2026-07-22,file-library creation date,Final consolidated verification target,final report,/mnt/data/archive/Unlearning_Project_Final_Results_Report.xlsx


In [7]:
def sha256_text(text: Any) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> Optional[str]:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_title(path: Path) -> Optional[str]:
    if not path.exists() or path.suffix.lower() != ".ipynb":
        return None
    try:
        import nbformat
        notebook = nbformat.read(path, as_version=4)
        for cell in notebook.cells:
            if cell.cell_type == "markdown":
                for line in cell.source.splitlines():
                    if line.strip().startswith("#"):
                        return line.lstrip("#").strip()
    except Exception:
        return None
    return None

def file_manifest_row(path: Path) -> dict[str, Any]:
    stat = path.stat() if path.exists() else None
    return {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": stat.st_size if stat else None,
        "modified_utc": (
            datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat()
            if stat else None
        ),
        "sha256": sha256_file(path),
        "notebook_title": notebook_title(path),
    }

details = ARTIFACT_REGISTRY["filename"].map(
    lambda name: file_manifest_row(ARCHIVE_DIR / name)
).apply(pd.Series)

ARTIFACT_MANIFEST = pd.concat([ARTIFACT_REGISTRY, details], axis=1)
display(ARTIFACT_MANIFEST[
    ["stage", "filename", "known_date", "date_basis", "exists", "sha256", "notebook_title"]
])

,stage,filename,known_date,date_basis,exists,sha256,notebook_title
0,demo,mini_unlearning_pipeline_api_pymupdf.ipynb,2026-05-15,file-library creation date,False,None,None
1,v2,Unlearning_AB_Test_Pipeline_v2.ipynb,2026-07-03,file-library creation date,False,None,None
2,v3,Unlearning_AB_Test_Pipeline_v3_fixed_api.ipynb,None,not recovered,False,None,None
3,v4,Unlearning_AB_Test_Pipeline_v4_PROVIDER_SEPARATED_FIXED.ipynb,None,run date not recovered,False,None,None
4,v5,Unlearning_Definitions_Only_Pipeline_Combined_Human_Test_Set.ipynb,2026-07-09,file-library creation date,False,None,None
5,v5_analysis,Unlearning_Weekly_Update_Review_Workbook.xlsx,2026-07-10,file-library creation date,False,None,None
6,v6,Unlearning_Local_Context_Definitions_Only_FIXED.ipynb,2026-07-17,file-library creation date,False,None,None
7,cleaned_audit,Unlearning_PostHoc_Cleaned_Metrics.xlsx,None,not recovered,False,None,None
8,production_v2,Unlearning_Production_MultiDocument_Annotation_Pipeline_v2_Verified_Extraction.ipynb,None,not recovered,False,None,None
9,final_report,Unlearning_Project_Final_Results_Report.xlsx,2026-07-22,file-library creation date,False,None,None


In [8]:
ARTIFACT_MANIFEST.to_csv(OUTPUTS_DIR / "historical_artifact_manifest.csv", index=False)
(OUTPUTS_DIR / "environment_snapshot.json").write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, indent=2, default=str), encoding="utf-8"
)
(OUTPUTS_DIR / "mega_notebook_config.json").write_text(
    json.dumps(asdict(CONFIG), indent=2, default=str), encoding="utf-8"
)
print("Initial provenance files saved in:", OUTPUTS_DIR)

Initial provenance files saved in: /mnt/data/outputs


# 4. Canonical schema and file loaders

In [9]:
def drop_empty_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy().dropna(axis=1, how="all")
    return result.loc[:, ~result.columns.astype(str).str.match(r"^Unnamed", na=False)]

def normalize_whitespace(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

def normalize_yes_no(value: Any) -> Optional[str]:
    text = normalize_whitespace(value).lower()
    if not text:
        return None
    if text in {"yes", "y", "true", "1", "unlearning", "present"} or text.startswith("yes"):
        return "Yes"
    if text in {"no", "n", "false", "0", "not unlearning", "non-unlearning", "absent"} or text.startswith("no"):
        return "No"
    return None

def label_to_int(value: Any) -> Optional[int]:
    return {"Yes": 1, "No": 0}.get(normalize_yes_no(value))

def normalized_text_fingerprint(value: Any) -> str:
    text = normalize_whitespace(value).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return sha256_text(text)

def dataframe_fingerprint(df: pd.DataFrame, columns: Optional[list[str]] = None) -> str:
    data = df.copy()
    if columns:
        missing = set(columns) - set(data.columns)
        if missing:
            raise KeyError(f"Missing fingerprint columns: {sorted(missing)}")
        data = data[columns]
    canonical = data.fillna("").astype(str).to_csv(index=False, lineterminator="\n")
    return sha256_text(canonical)

def read_table(path: Path, sheet_name: Any = 0) -> pd.DataFrame:
    if not path.exists():
        if CONFIG.strict_missing_files:
            raise FileNotFoundError(path)
        print(f"SKIP: {path} not found")
        return pd.DataFrame()

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".jsonl":
        return pd.DataFrame([
            json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ])
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {suffix}")

In [10]:
REQUIRED_PREDICTION_COLUMNS = {
    "row_id", "provider", "model", "prompt_variant",
    "ground_truth_unlearning", "pred_unlearning", "status"
}

def canonicalize_predictions(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    result = df.copy()
    aliases = {
        "Number": "row_id",
        "provider_name": "provider",
        "model_name": "model",
        "prompt": "prompt_variant",
        "ground_truth": "ground_truth_unlearning",
        "human_label": "ground_truth_unlearning",
        "prediction": "pred_unlearning",
        "predicted_unlearning": "pred_unlearning",
        "Document": "document",
        "Text Content": "text",
    }
    for old, new in aliases.items():
        if old in result.columns and new not in result.columns:
            result = result.rename(columns={old: new})

    if "status" not in result.columns:
        result["status"] = "ok"

    missing = REQUIRED_PREDICTION_COLUMNS - set(result.columns)
    if missing:
        raise ValueError(f"Missing prediction columns: {sorted(missing)}")

    result["row_id"] = result["row_id"].astype(str)
    result["ground_truth_unlearning"] = result["ground_truth_unlearning"].map(normalize_yes_no)
    result["pred_unlearning"] = result["pred_unlearning"].map(normalize_yes_no)
    return result

# 5. Experiment registry

In [11]:
EXPERIMENT_REGISTRY = pd.DataFrame([
    {
        "experiment_id": "v2_early_pilot",
        "test_n": 11,
        "gold_distribution": "11 Yes / 0 No",
        "codebook": "earlier",
        "design": "four prompt variants; OpenAI complete; Gemini partial",
        "primary_metric": "positive recall",
        "comparison_boundary": "within the same 11 rows and prompt construction",
    },
    {
        "experiment_id": "v3_prompt_parity",
        "test_n": 11,
        "gold_distribution": "11 Yes / 0 No",
        "codebook": "earlier",
        "design": "one controlled model × four prompts; 44/44 calls",
        "primary_metric": "positive recall",
        "comparison_boundary": "conditions within v3",
    },
    {
        "experiment_id": "v4_provider_grid",
        "test_n": 11,
        "gold_distribution": "11 Yes / 0 No",
        "codebook": "earlier",
        "design": "provider-separated model grid",
        "primary_metric": "positive recall",
        "comparison_boundary": "complete model groups on identical rows",
    },
    {
        "experiment_id": "v5_definitions_mixed",
        "test_n": 49,
        "gold_distribution": "39 Yes / 10 No",
        "codebook": "current/stricter",
        "design": "five models; definitions only; 245/245 calls",
        "primary_metric": "accuracy, precision, recall, specificity, F1, kappa",
        "comparison_boundary": "models inside v5",
    },
    {
        "experiment_id": "v6_local_context",
        "test_n": 42,
        "gold_distribution": "32 Yes / 10 No",
        "codebook": "current/stricter",
        "design": "three providers; definitions plus local context; 126/126 calls",
        "primary_metric": "accuracy, recall, specificity, F1, document recall",
        "comparison_boundary": "providers inside v6",
    },
    {
        "experiment_id": "cleaned_provider_strategy",
        "test_n": "40–46 by strategy",
        "gold_distribution": "mixed",
        "codebook": "historical prompt-grid data",
        "design": "duplicates removed; few-shot overlap excluded",
        "primary_metric": "cleaned accuracy, F1, kappa, confusion counts",
        "comparison_boundary": "same cleaning rule and denominator",
    },
    {
        "experiment_id": "production_verified_article",
        "test_n": 32,
        "gold_distribution": "no human gold",
        "codebook": "current",
        "design": "three provider coders; 96 predictions; verified extraction",
        "primary_metric": "agreement, reliability, consensus, review queue",
        "comparison_boundary": "not comparable to benchmark accuracy",
    },
])
display(EXPERIMENT_REGISTRY)

,experiment_id,test_n,gold_distribution,codebook,design,primary_metric,comparison_boundary
0,v2_early_pilot,11,11 Yes / 0 No,earlier,four prompt variants; OpenAI complete; Gemini partial,positive recall,within the same 11 rows and prompt construction
1,v3_prompt_parity,11,11 Yes / 0 No,earlier,one controlled model × four prompts; 44/44 calls,positive recall,conditions within v3
2,v4_provider_grid,11,11 Yes / 0 No,earlier,provider-separated model grid,positive recall,complete model groups on identical rows
3,v5_definitions_mixed,49,39 Yes / 10 No,current/stricter,five models; definitions only; 245/245 calls,"accuracy, precision, recall, specificity, F1, kappa",models inside v5
4,v6_local_context,42,32 Yes / 10 No,current/stricter,three providers; definitions plus local context; 126/126 calls,"accuracy, recall, specificity, F1, document recall",providers inside v6
5,cleaned_provider_strategy,40–46 by strategy,mixed,historical prompt-grid data,duplicates removed; few-shot overlap excluded,"cleaned accuracy, F1, kappa, confusion counts",same cleaning rule and denominator
6,production_verified_article,32,no human gold,current,three provider coders; 96 predictions; verified extraction,"agreement, reliability, consensus, review queue",not comparable to benchmark accuracy


# 6. Historical result paths

In [12]:
# Update these names after placing the exact raw files in archive/.
HISTORICAL_RESULT_PATHS = {
    "v2_predictions": ARCHIVE_DIR / "predictions_long.csv",
    "v3_predictions": ARCHIVE_DIR / "predictions_prompt_parity.csv",
    "v4_predictions": ARCHIVE_DIR / "predictions_v4_provider_separated.csv",
    "v5_predictions": ARCHIVE_DIR / "predictions_long_v5_definitions_only_combined_human_test.csv",
    "v6_predictions": ARCHIVE_DIR / "local_context_predictions.csv",
    "production_predictions": ARCHIVE_DIR / "production_provider_predictions.csv",
}

HISTORICAL_TABLES = {
    name: read_table(path)
    for name, path in HISTORICAL_RESULT_PATHS.items()
}

display(pd.DataFrame([
    {"name": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in HISTORICAL_TABLES.items()
]))

SKIP: /mnt/data/archive/predictions_long.csv not found
SKIP: /mnt/data/archive/predictions_prompt_parity.csv not found
SKIP: /mnt/data/archive/predictions_v4_provider_separated.csv not found
SKIP: /mnt/data/archive/predictions_long_v5_definitions_only_combined_human_test.csv not found
SKIP: /mnt/data/archive/local_context_predictions.csv not found
SKIP: /mnt/data/archive/production_provider_predictions.csv not found


,name,rows,columns
0,v2_predictions,0,0
1,v3_predictions,0,0
2,v4_predictions,0,0
3,v5_predictions,0,0
4,v6_predictions,0,0
5,production_predictions,0,0


# 7. Binary classification analysis

In [13]:
def binary_metrics(y_true: Iterable[Any], y_pred: Iterable[Any]) -> dict[str, Any]:
    pairs = [
        (label_to_int(t), label_to_int(p))
        for t, p in zip(y_true, y_pred)
    ]
    pairs = [(t, p) for t, p in pairs if t is not None and p is not None]
    if not pairs:
        return {}

    yt = np.array([x[0] for x in pairs], dtype=int)
    yp = np.array([x[1] for x in pairs], dtype=int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

    return {
        "n": len(yt),
        "human_yes": int(yt.sum()),
        "human_no": int((1 - yt).sum()),
        "pred_yes": int(yp.sum()),
        "pred_no": int((1 - yp).sum()),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "accuracy": accuracy_score(yt, yp),
        "precision_yes": precision_score(yt, yp, zero_division=0),
        "recall_yes": recall_score(yt, yp, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1_yes": f1_score(yt, yp, zero_division=0),
        "cohen_kappa": cohen_kappa_score(yt, yp),
    }

def summarize_groups(
    df: pd.DataFrame,
    group_columns: list[str],
    gold_column: str = "ground_truth_unlearning",
    pred_column: str = "pred_unlearning",
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    data = canonicalize_predictions(df)
    data = data[data["status"].eq("ok")].copy()
    rows = []

    for keys, group in data.groupby(group_columns, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(binary_metrics(group[gold_column], group[pred_column]))

        for col in ["input_tokens", "output_tokens", "total_tokens", "cost_usd", "latency_seconds"]:
            if col in group.columns:
                numeric = pd.to_numeric(group[col], errors="coerce")
                row[f"{col}_sum"] = numeric.sum()
                row[f"{col}_mean"] = numeric.mean()
        rows.append(row)

    return pd.DataFrame(rows)

In [14]:
def benchmark_class_check(labels: Iterable[Any]) -> dict[str, Any]:
    normalized = pd.Series(labels).map(normalize_yes_no).dropna()
    counts = normalized.value_counts().to_dict()
    classes = sorted(counts)
    return {
        "n": len(normalized),
        "class_counts": counts,
        "single_class_benchmark": len(classes) < 2,
        "recommended_metric": (
            "positive recall only" if classes == ["Yes"]
            else "negative recall/specificity only" if classes == ["No"]
            else "accuracy, precision, recall, specificity, F1, kappa"
        ),
    }

def error_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    data = canonicalize_predictions(df)
    data["correct"] = data["ground_truth_unlearning"].eq(data["pred_unlearning"])
    data["error_type"] = np.select(
        [
            data["ground_truth_unlearning"].eq("Yes") & data["pred_unlearning"].eq("No"),
            data["ground_truth_unlearning"].eq("No") & data["pred_unlearning"].eq("Yes"),
            data["correct"],
        ],
        ["False Negative", "False Positive", "Correct"],
        default="Unscorable",
    )
    return data

def document_metrics(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    data = canonicalize_predictions(df)
    if "document" not in data.columns:
        data["document"] = "Unknown document"
    return summarize_groups(
        data, ["provider", "model", "prompt_variant", "document"]
    )

# 8. Duplicate and leakage audit

In [15]:
def exact_duplicates(
    df: pd.DataFrame,
    text_column: str = CONFIG.text_column,
    id_column: str = CONFIG.id_column,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    data = df[[id_column, text_column]].copy()
    data["text_sha256"] = data[text_column].map(normalized_text_fingerprint)
    return data[data.duplicated("text_sha256", keep=False)].sort_values(
        ["text_sha256", id_column]
    )

def exact_example_test_overlap(
    examples: pd.DataFrame,
    test: pd.DataFrame,
    example_text_column: str = CONFIG.text_column,
    test_text_column: str = CONFIG.text_column,
    example_id_column: str = CONFIG.id_column,
    test_id_column: str = CONFIG.id_column,
) -> pd.DataFrame:
    if examples.empty or test.empty:
        return pd.DataFrame()

    left = test[[test_id_column, test_text_column]].copy()
    right = examples[[example_id_column, example_text_column]].copy()
    left["text_sha256"] = left[test_text_column].map(normalized_text_fingerprint)
    right["text_sha256"] = right[example_text_column].map(normalized_text_fingerprint)

    return left.merge(
        right, on="text_sha256", how="inner", suffixes=("_test", "_example")
    )

def token_jaccard(a: Any, b: Any) -> float:
    ta = set(re.findall(r"[a-z0-9]+", normalize_whitespace(a).lower()))
    tb = set(re.findall(r"[a-z0-9]+", normalize_whitespace(b).lower()))
    return len(ta & tb) / len(ta | tb) if (ta | tb) else 1.0

def near_duplicates_small_set(
    df: pd.DataFrame,
    text_column: str = CONFIG.text_column,
    id_column: str = CONFIG.id_column,
    threshold: float = 0.85,
) -> pd.DataFrame:
    # O(n²): intended only for small benchmark sheets.
    records = df[[id_column, text_column]].dropna().to_dict("records")
    pairs = []
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            score = token_jaccard(records[i][text_column], records[j][text_column])
            if score >= threshold:
                pairs.append({
                    "id_a": records[i][id_column],
                    "id_b": records[j][id_column],
                    "jaccard": score,
                    "text_a": records[i][text_column],
                    "text_b": records[j][text_column],
                })
    return pd.DataFrame(pairs).sort_values("jaccard", ascending=False) if pairs else pd.DataFrame()

# 9. Prompt and schema provenance

In [16]:
PROMPT_SCHEMA_VERSION = "unlearning_annotation_schema_v1"

OUTPUT_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "unlearning_present", "target_type", "agency", "confidence", "rationale"
    ],
    "properties": {
        "unlearning_present": {"type": "boolean"},
        "target_type": {
            "type": "string",
            "enum": [
                "leadership", "laws_plans_policies", "capabilities",
                "funds_resources", "misc_organizational", "none"
            ],
        },
        "agency": {"type": ["string", "null"]},
        "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "rationale": {"type": "string"},
    },
}

PROMPT_VARIANTS = pd.DataFrame([
    ["direct_no_context", False, False, False, False],
    ["definitions_only", True, False, False, False],
    ["definitions_examples_no_metadata", True, True, False, False],
    ["definitions_examples_with_metadata", True, True, True, False],
    ["definitions_local_context", True, False, True, True],
], columns=[
    "prompt_variant", "uses_codebook", "uses_examples",
    "uses_metadata", "uses_local_context"
])

display(PROMPT_VARIANTS)

,prompt_variant,uses_codebook,uses_examples,uses_metadata,uses_local_context
0,direct_no_context,False,False,False,False
1,definitions_only,True,False,False,False
2,definitions_examples_no_metadata,True,True,False,False
3,definitions_examples_with_metadata,True,True,True,False
4,definitions_local_context,True,False,True,True


In [17]:
def render_prompt(
    prompt_variant: str,
    row_id: Any,
    target_text: str,
    codebook_text: str = "",
    examples_text: str = "",
    metadata_text: str = "",
    local_context_text: str = "",
) -> dict[str, str]:
    match = PROMPT_VARIANTS[
        PROMPT_VARIANTS["prompt_variant"].eq(prompt_variant)
    ]
    if match.empty:
        raise ValueError(f"Unknown prompt variant: {prompt_variant}")
    spec = match.iloc[0]

    sections = [
        "You are coding federal disaster-policy text for organizational unlearning.",
        (
            "Classify the TARGET PARAGRAPH. Return a binary decision, one target "
            "category when positive, agency if stated, confidence, and a short rationale."
        ),
    ]

    if spec["uses_codebook"]:
        sections.append("CODEBOOK:\n" + codebook_text.strip())
    if spec["uses_examples"]:
        sections.append("LABELED EXAMPLES:\n" + examples_text.strip())
    if spec["uses_metadata"] and metadata_text.strip():
        sections.append("TARGET METADATA:\n" + metadata_text.strip())
    if spec["uses_local_context"] and local_context_text.strip():
        sections.append(
            "Use local context only to interpret the target. Do not transfer a "
            "neighboring paragraph's label to the target."
        )
        sections.append("LOCAL CONTEXT:\n" + local_context_text.strip())

    sections += [
        f"PARAGRAPH ID: {row_id}",
        "TARGET PARAGRAPH:\n" + normalize_whitespace(target_text),
        f"OUTPUT SCHEMA VERSION: {PROMPT_SCHEMA_VERSION}",
        "Return valid JSON only.",
    ]

    rendered = "\n\n".join(x for x in sections if x.strip())
    return {
        "prompt_variant": prompt_variant,
        "prompt_text": rendered,
        "prompt_sha256": sha256_text(rendered),
        "schema_sha256": sha256_text(json.dumps(OUTPUT_SCHEMA, sort_keys=True)),
    }

preview = render_prompt(
    "direct_no_context", "demo",
    "FEMA replaced an earlier planning assumption after Katrina."
)
print(preview["prompt_sha256"])
print(preview["prompt_text"][:700])

d2a84145437d47d12309a35ce498ccf0b03d5d37886f0e266dbd02066c8dc510
You are coding federal disaster-policy text for organizational unlearning.

Classify the TARGET PARAGRAPH. Return a binary decision, one target category when positive, agency if stated, confidence, and a short rationale.

PARAGRAPH ID: demo

TARGET PARAGRAPH:
FEMA replaced an earlier planning assumption after Katrina.

OUTPUT SCHEMA VERSION: unlearning_annotation_schema_v1

Return valid JSON only.


# 10. Historical model registry

In [18]:
MODEL_REGISTRY = pd.DataFrame([
    ["v2_openai", "openai", "gpt-5.4-nano", "v2", 0.0, None,
     "Retained from historical v2 configuration"],
    ["v2_anthropic", "anthropic", "claude-haiku-4-5-20251001", "v2/v5", 0.0, None,
     "Retained from historical configuration"],
    ["v2_google", "google", "gemini-2.5-flash-lite", "v2", 0.0, None,
     "Retained from historical v2 configuration"],
    ["v5_openai", "openai", "gpt-5.4", "v5", 0.0, "low",
     "Verify exact returned snapshot from saved response metadata"],
    ["v5_google", "google", "gemini-3.5-flash", "v5", 0.0, "low",
     "Verify exact returned snapshot from saved response metadata"],
], columns=[
    "config_id", "provider", "model", "historical_stage",
    "temperature", "reasoning_or_thinking", "notes"
])
display(MODEL_REGISTRY)

,config_id,provider,model,historical_stage,temperature,reasoning_or_thinking,notes
0,v2_openai,openai,gpt-5.4-nano,v2,0.0,None,Retained from historical v2 configuration
1,v2_anthropic,anthropic,claude-haiku-4-5-20251001,v2/v5,0.0,None,Retained from historical configuration
2,v2_google,google,gemini-2.5-flash-lite,v2,0.0,None,Retained from historical v2 configuration
3,v5_openai,openai,gpt-5.4,v5,0.0,low,Verify exact returned snapshot from saved response metadata
4,v5_google,google,gemini-3.5-flash,v5,0.0,low,Verify exact returned snapshot from saved response metadata


# 11. Controlled run logging

This section is provider-agnostic. Provider SDK calls should remain in small adapters so an SDK update does not alter the experiment logic.

In [19]:
@dataclass
class RunContext:
    experiment_id: str
    prompt_version: str
    dataset_sha256: str
    codebook_sha256: str
    config: dict[str, Any]
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex)
    started_at_utc: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    @property
    def run_dir(self) -> Path:
        return RUNS_DIR / self.experiment_id / self.run_id

def create_run_context(
    experiment_id: str,
    prompt_version: str,
    dataset_df: pd.DataFrame,
    codebook_text: str,
    extra_config: Optional[dict[str, Any]] = None,
) -> RunContext:
    context = RunContext(
        experiment_id=experiment_id,
        prompt_version=prompt_version,
        dataset_sha256=dataframe_fingerprint(dataset_df),
        codebook_sha256=sha256_text(codebook_text),
        config={**asdict(CONFIG), **(extra_config or {})},
    )
    context.run_dir.mkdir(parents=True, exist_ok=False)
    (context.run_dir / "run_context.json").write_text(
        json.dumps(asdict(context), indent=2, default=str), encoding="utf-8"
    )
    return context

def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

def deterministic_run_key(payload: dict[str, Any]) -> str:
    return sha256_text(json.dumps(payload, sort_keys=True, default=str))

In [20]:
# A future provider adapter must return normalized fields such as:
# pred_unlearning, target_type, agency, confidence, rationale,
# raw_response_text, input_tokens, output_tokens, total_tokens,
# cost_usd, returned_model, and request_id.

ProviderCall = Callable[[dict[str, Any], dict[str, Any]], dict[str, Any]]

def run_rows(
    rows: pd.DataFrame,
    context: RunContext,
    provider_config: dict[str, Any],
    prompt_variant: str,
    build_prompt: Callable[[pd.Series], dict[str, str]],
    provider_call: ProviderCall,
    id_column: str = CONFIG.id_column,
    gold_column: str = CONFIG.gold_column,
) -> pd.DataFrame:
    if not CONFIG.run_api_calls:
        raise RuntimeError(
            "API calls are disabled. Create a new controlled experiment before enabling them."
        )

    results_path = context.run_dir / "raw_results.jsonl"
    existing = read_table(results_path)
    existing_keys = set(existing.get("run_key", pd.Series(dtype=str)).dropna())
    records = existing.to_dict("records")

    for _, row in rows.iterrows():
        prompt = build_prompt(row)
        row_id = str(row[id_column])
        key_payload = {
            "experiment_id": context.experiment_id,
            "provider": provider_config["provider"],
            "model": provider_config["model"],
            "prompt_variant": prompt_variant,
            "row_id": row_id,
            "prompt_sha256": prompt["prompt_sha256"],
            "dataset_sha256": context.dataset_sha256,
            "codebook_sha256": context.codebook_sha256,
        }
        run_key = deterministic_run_key(key_payload)
        if run_key in existing_keys:
            continue

        started = time.perf_counter()
        base = {
            **key_payload,
            "run_id": context.run_id,
            "run_key": run_key,
            "prompt_version": context.prompt_version,
            "schema_sha256": prompt["schema_sha256"],
            "ground_truth_unlearning": normalize_yes_no(row.get(gold_column)),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }

        try:
            response = provider_call(provider_config, {
                "prompt_text": prompt["prompt_text"],
                "output_schema": OUTPUT_SCHEMA,
                "temperature": CONFIG.temperature,
                "max_output_tokens": CONFIG.max_output_tokens,
            })
            record = {
                **base, **response, "status": "ok",
                "latency_seconds": time.perf_counter() - started,
                "response_sha256": sha256_text(response.get("raw_response_text", "")),
            }
        except Exception as exc:
            record = {
                **base, "status": "error",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "latency_seconds": time.perf_counter() - started,
            }

        append_jsonl(results_path, record)
        records.append(record)
        existing_keys.add(run_key)
        time.sleep(CONFIG.request_sleep_seconds)

    return pd.DataFrame(records)

def provider_adapter_not_configured(*args, **kwargs):
    raise NotImplementedError(
        "This audit notebook intentionally contains no enabled paid-call adapter."
    )

# 12. Cost and usage audit

In [21]:
def usage_summary(
    df: pd.DataFrame,
    group_columns: list[str] = ["provider", "model", "prompt_variant"],
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    data = df.copy()
    for col in ["input_tokens", "output_tokens", "total_tokens", "cost_usd", "latency_seconds"]:
        if col not in data.columns:
            data[col] = np.nan
        data[col] = pd.to_numeric(data[col], errors="coerce")

    return (
        data.groupby(group_columns, dropna=False)
        .agg(
            rows=("row_id", "count"),
            successful_rows=("status", lambda x: int(pd.Series(x).eq("ok").sum())),
            input_tokens=("input_tokens", "sum"),
            output_tokens=("output_tokens", "sum"),
            total_tokens=("total_tokens", "sum"),
            cost_usd=("cost_usd", "sum"),
            average_latency_seconds=("latency_seconds", "mean"),
        )
        .reset_index()
    )

def check_budget(estimated_cost_usd: float) -> None:
    if estimated_cost_usd > CONFIG.max_estimated_cost_usd:
        raise RuntimeError(
            f"Estimated cost ${estimated_cost_usd:.4f} exceeds "
            f"${CONFIG.max_estimated_cost_usd:.2f}."
        )

# 13. Consensus and inter-coder reliability

In [22]:
def majority_label(values: Iterable[Any]) -> Optional[str]:
    labels = [normalize_yes_no(v) for v in values]
    labels = [v for v in labels if v is not None]
    if not labels:
        return None
    counts = Counter(labels)
    ordered = counts.most_common()
    if len(ordered) > 1 and ordered[0][1] == ordered[1][1]:
        return None
    return ordered[0][0]

def consensus_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    rows = []
    for row_id, group in df.groupby("row_id", dropna=False):
        labels = group["pred_unlearning"].map(normalize_yes_no).dropna().tolist()
        counts = Counter(labels)
        consensus = majority_label(labels)
        rows.append({
            "row_id": row_id,
            "n_coders": len(labels),
            "yes_votes": counts.get("Yes", 0),
            "no_votes": counts.get("No", 0),
            "consensus": consensus,
            "unanimous": len(counts) == 1 and bool(labels),
            "agreement_share": max(counts.values()) / len(labels) if labels else np.nan,
            "review_priority": (
                "high" if len(counts) > 1
                else "medium" if consensus == "Yes"
                else "low"
            ),
        })
    return pd.DataFrame(rows)

def binary_reliability(df: pd.DataFrame) -> dict[str, Any]:
    if df.empty:
        return {}

    pivot = df.pivot_table(
        index="row_id", columns="provider",
        values="pred_unlearning", aggfunc="first"
    )
    encoded = pivot.apply(lambda col: col.map(label_to_int))
    complete = encoded.dropna()

    output = {
        "rows_total": len(encoded),
        "rows_complete": len(complete),
        "providers": list(encoded.columns),
        "unanimous_agreement": (
            complete.nunique(axis=1).eq(1).mean() if not complete.empty else np.nan
        ),
    }

    if len(complete.columns) == 2:
        output["cohen_kappa"] = cohen_kappa_score(
            complete.iloc[:, 0], complete.iloc[:, 1]
        )

    if len(complete.columns) >= 3 and fleiss_kappa is not None:
        counts = np.column_stack([
            (complete == 0).sum(axis=1),
            (complete == 1).sum(axis=1),
        ])
        output["fleiss_kappa"] = float(fleiss_kappa(counts))

    if len(complete.columns) >= 2 and krippendorff is not None:
        output["krippendorff_alpha"] = float(
            krippendorff.alpha(
                reliability_data=complete.to_numpy().T,
                level_of_measurement="nominal",
            )
        )

    return output

# 14. Verification targets

In [23]:
VERIFICATION_TARGETS = pd.DataFrame([
    ["v2", "design", "11 Yes-only passages; OpenAI complete across four prompts"],
    ["v3", "completion", "44 / 44 calls"],
    ["v4", "Claude definitions-only", "11 TP, 0 FN; positive recall = 1.00"],
    ["v5", "test composition", "49 passages: 39 Yes, 10 No; 245 / 245 calls"],
    ["v5", "review totals", "122 false-negative predictions; 14 unanimous FN rows"],
    ["v5", "recorded cost", "approximately $0.82272"],
    ["v6", "test composition", "42 passages: 32 Yes, 10 No; 126 / 126 calls"],
    ["cleaned", "provider comparison", "OpenAI best cleaned accuracy; Gemini best cleaned F1"],
    ["production", "volume", "32 logical paragraphs; 96 predictions; 18 annotation segments"],
    ["production", "binary reliability", "agreement 0.6875; Fleiss kappa ≈ 0.485; alpha ≈ 0.490"],
], columns=["stage", "measure", "expected_value"])
display(VERIFICATION_TARGETS)

,stage,measure,expected_value
0,v2,design,11 Yes-only passages; OpenAI complete across four prompts
1,v3,completion,44 / 44 calls
2,v4,Claude definitions-only,"11 TP, 0 FN; positive recall = 1.00"
3,v5,test composition,"49 passages: 39 Yes, 10 No; 245 / 245 calls"
4,v5,review totals,122 false-negative predictions; 14 unanimous FN rows
5,v5,recorded cost,approximately $0.82272
6,v6,test composition,"42 passages: 32 Yes, 10 No; 126 / 126 calls"
7,cleaned,provider comparison,OpenAI best cleaned accuracy; Gemini best cleaned F1
8,production,volume,32 logical paragraphs; 96 predictions; 18 annotation segments
9,production,binary reliability,agreement 0.6875; Fleiss kappa ≈ 0.485; alpha ≈ 0.490


# 15. Document-level and difficult-row analysis

In [24]:
def source_error_profile(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    data = error_table(df)
    if "document" not in data.columns:
        data["document"] = "Unknown document"

    profile = (
        data.groupby(["provider", "model", "prompt_variant", "document"], dropna=False)
        .agg(
            rows=("row_id", "count"),
            human_yes=("ground_truth_unlearning", lambda x: int(pd.Series(x).eq("Yes").sum())),
            human_no=("ground_truth_unlearning", lambda x: int(pd.Series(x).eq("No").sum())),
            false_negatives=("error_type", lambda x: int(pd.Series(x).eq("False Negative").sum())),
            false_positives=("error_type", lambda x: int(pd.Series(x).eq("False Positive").sum())),
            correct=("error_type", lambda x: int(pd.Series(x).eq("Correct").sum())),
        )
        .reset_index()
    )
    profile["positive_recall"] = np.where(
        profile["human_yes"] > 0,
        (profile["human_yes"] - profile["false_negatives"]) / profile["human_yes"],
        np.nan,
    )
    profile["specificity"] = np.where(
        profile["human_no"] > 0,
        (profile["human_no"] - profile["false_positives"]) / profile["human_no"],
        np.nan,
    )
    return profile

def difficult_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    data = error_table(df)
    groups = ["row_id", "ground_truth_unlearning"]
    if "document" in data.columns:
        groups.append("document")
    if "text" in data.columns:
        groups.append("text")

    result = (
        data.groupby(groups, dropna=False)
        .agg(
            n_predictions=("pred_unlearning", "count"),
            n_correct=("correct", "sum"),
            false_negatives=("error_type", lambda x: int(pd.Series(x).eq("False Negative").sum())),
            false_positives=("error_type", lambda x: int(pd.Series(x).eq("False Positive").sum())),
            predicted_yes=("pred_unlearning", lambda x: int(pd.Series(x).eq("Yes").sum())),
            predicted_no=("pred_unlearning", lambda x: int(pd.Series(x).eq("No").sum())),
        )
        .reset_index()
    )
    result["error_rate"] = 1 - result["n_correct"] / result["n_predictions"]
    return result.sort_values(
        ["error_rate", "false_negatives"], ascending=[False, False]
    )

# 16. Old and new codebook record

In [25]:
CODEBOOK_COMPARISON = pd.DataFrame([
    [
        "Core question",
        "Does the passage show unlearning through reconsidering, discarding, realignment, merging, or related change?",
        "Does the passage identify an earlier logic/practice as inadequate and show departure from it?"
    ],
    [
        "Positive threshold",
        "Broad enough to include integration, adaptation, and technical realignment.",
        "Requires clearer rejection, replacement, removal, abandonment, or fundamental rethinking."
    ],
    [
        "Additive improvement",
        "Could count when it represented meaningful institutional adaptation.",
        "Usually learning/reform unless the earlier approach being displaced is identified."
    ],
    [
        "Likely metric effect",
        "Higher sensitivity to implicit and incremental change.",
        "Lower recall against broad legacy labels but a sharper learning/unlearning boundary."
    ],
], columns=["dimension", "older_framework", "newer_framework"])
display(CODEBOOK_COMPARISON)

,dimension,older_framework,newer_framework
0,Core question,"Does the passage show unlearning through reconsidering, discarding, realignment, merging, or related change?",Does the passage identify an earlier logic/practice as inadequate and show departure from it?
1,Positive threshold,"Broad enough to include integration, adaptation, and technical realignment.","Requires clearer rejection, replacement, removal, abandonment, or fundamental rethinking."
2,Additive improvement,Could count when it represented meaningful institutional adaptation.,Usually learning/reform unless the earlier approach being displaced is identified.
3,Likely metric effect,Higher sensitivity to implicit and incremental change.,Lower recall against broad legacy labels but a sharper learning/unlearning boundary.


# 17. Extraction gate and production verification

In [26]:
@dataclass
class ExtractionCheck:
    name: str
    passed: bool
    observed: Any
    requirement: str
    notes: str = ""

def extraction_gate(checks: list[ExtractionCheck], human_approved: bool) -> None:
    failed = [check for check in checks if not check.passed]
    if failed:
        details = "\n".join(
            f"- {x.name}: observed={x.observed}; required={x.requirement}"
            for x in failed
        )
        raise RuntimeError("Extraction checks failed:\n" + details)
    if not human_approved:
        raise RuntimeError("Automated checks passed, but human approval is False.")
    print("Extraction gate passed.")

HISTORICAL_PRODUCTION_CHECKS = [
    ExtractionCheck("logical paragraph count", True, 32,
                    "both extractors reconstruct the same logical units"),
    ExtractionCheck("cross-extractor normalized similarity", True, 1.0,
                    "minimum aligned similarity = 1.0 in verified run"),
    ExtractionCheck("minimum annotation text similarity", True, 0.843188,
                    "minimum >= 0.80"),
    ExtractionCheck("annotation segments", True, "18 / 18",
                    "all expected segments verified"),
]
display(pd.DataFrame([asdict(x) for x in HISTORICAL_PRODUCTION_CHECKS]))

,name,passed,observed,requirement,notes
0,logical paragraph count,True,32,both extractors reconstruct the same logical units,
1,cross-extractor normalized similarity,True,1.0,minimum aligned similarity = 1.0 in verified run,
2,minimum annotation text similarity,True,0.843188,minimum >= 0.80,
3,annotation segments,True,18 / 18,all expected segments verified,


# 18. Disagreement review queue

In [27]:
def build_review_queue(
    predictions: pd.DataFrame,
    text_table: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()

    data = canonicalize_predictions(predictions)
    queue = consensus_table(data)
    queue = queue[
        queue["review_priority"].eq("high") | queue["consensus"].eq("Yes")
    ].copy()

    if text_table is not None and not text_table.empty:
        source = text_table.copy()
        source_id = CONFIG.id_column if CONFIG.id_column in source.columns else "row_id"
        source[source_id] = source[source_id].astype(str)
        queue["row_id"] = queue["row_id"].astype(str)
        keep = [
            col for col in [
                source_id, CONFIG.text_column, CONFIG.document_column,
                "PDF Page", "Section Heading", CONFIG.context_column, "Rationale"
            ]
            if col in source.columns
        ]
        queue = queue.merge(
            source[keep].drop_duplicates(source_id),
            left_on="row_id", right_on=source_id, how="left"
        )

    detail_columns = [
        col for col in [
            "row_id", "provider", "model", "pred_unlearning",
            "target_type", "agency", "confidence", "rationale"
        ]
        if col in data.columns
    ]
    details = data[detail_columns].copy()
    details["model_output"] = details.apply(
        lambda row: " | ".join(
            f"{col}={row[col]}" for col in detail_columns if col != "row_id"
        ), axis=1
    )
    rollup = (
        details.groupby("row_id")["model_output"]
        .apply(lambda values: "\n".join(values))
        .rename("model_outputs").reset_index()
    )
    return queue.merge(rollup, on="row_id", how="left")

# 19. Generate the historical audit

In [28]:
def generate_audit_reports(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    reports = {}
    for name, raw in tables.items():
        if raw.empty:
            continue
        try:
            data = canonicalize_predictions(raw)
        except Exception as exc:
            reports[name + "__schema_error"] = pd.DataFrame([{
                "table": name,
                "error": str(exc),
                "columns": ", ".join(map(str, raw.columns)),
            }])
            continue

        reports[name + "__metrics"] = summarize_groups(
            data, ["provider", "model", "prompt_variant"]
        )
        reports[name + "__documents"] = document_metrics(data)
        reports[name + "__source_errors"] = source_error_profile(data)
        reports[name + "__difficult_rows"] = difficult_rows(data)
        reports[name + "__usage"] = usage_summary(data)

    return reports

AUDIT_REPORTS = generate_audit_reports(HISTORICAL_TABLES)

for name, report in AUDIT_REPORTS.items():
    print("\n" + "=" * 100)
    print(name)
    display(report.head(25))

In [29]:
AUDIT_XLSX = REPORTS_DIR / "mega_notebook_historical_audit.xlsx"

if AUDIT_REPORTS:
    with pd.ExcelWriter(AUDIT_XLSX, engine="openpyxl") as writer:
        ARTIFACT_REGISTRY.to_excel(writer, sheet_name="Artifact Registry", index=False)
        EXPERIMENT_REGISTRY.to_excel(writer, sheet_name="Experiment Registry", index=False)
        VERIFICATION_TARGETS.to_excel(writer, sheet_name="Verification Targets", index=False)
        CODEBOOK_COMPARISON.to_excel(writer, sheet_name="Codebook Comparison", index=False)

        used_names = set()
        for name, report in AUDIT_REPORTS.items():
            base = re.sub(r"[^A-Za-z0-9 _-]", "_", name)[:31]
            sheet = base
            number = 1
            while sheet in used_names:
                suffix = f"_{number}"
                sheet = (base[:31-len(suffix)] + suffix)
                number += 1
            used_names.add(sheet)
            report.to_excel(writer, sheet_name=sheet, index=False)

    print("Saved:", AUDIT_XLSX)
else:
    print("No historical prediction files found. Add them to archive/ and rerun.")

No historical prediction files found. Add them to archive/ and rerun.


# 20. Freeze a verified run bundle

In [30]:
def copy_if_exists(source: Path, destination: Path) -> Optional[Path]:
    if not source.exists():
        return None
    target = destination / source.name
    shutil.copy2(source, target)
    return target

def export_verified_bundle(
    bundle_name: str,
    files: Iterable[Path],
    tables: Optional[dict[str, pd.DataFrame]] = None,
    notes: str = "",
) -> Path:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    bundle = OUTPUTS_DIR / "verified_bundles" / f"{bundle_name}_{stamp}"
    bundle.mkdir(parents=True, exist_ok=False)

    copied = []
    for source in files:
        target = copy_if_exists(Path(source), bundle)
        if target:
            copied.append(target.name)

    table_records = []
    for name, df in (tables or {}).items():
        path = bundle / f"{name}.csv"
        df.to_csv(path, index=False)
        table_records.append({
            "name": name,
            "filename": path.name,
            "rows": len(df),
            "sha256": sha256_file(path),
        })

    manifest = pd.DataFrame([
        file_manifest_row(path) for path in bundle.iterdir() if path.is_file()
    ])
    manifest.to_csv(bundle / "bundle_file_manifest.csv", index=False)

    metadata = {
        "bundle_name": bundle_name,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook_version": CONFIG.notebook_version,
        "config": asdict(CONFIG),
        "environment": ENVIRONMENT_SNAPSHOT,
        "copied_files": copied,
        "tables": table_records,
        "notes": notes,
    }
    (bundle / "bundle_metadata.json").write_text(
        json.dumps(metadata, indent=2, default=str), encoding="utf-8"
    )
    return bundle

# Example after verification:
# export_verified_bundle(
#     "v5_definitions_verified",
#     files=[HISTORICAL_RESULT_PATHS["v5_predictions"]],
#     tables={"metrics": AUDIT_REPORTS["v5_predictions__metrics"]},
#     notes="Verified against the consolidated report."
# )

# 21. Transition to version control

Recommended repository structure:

```text
unlearning-project/
├── README.md
├── pyproject.toml
├── .gitignore
├── notebooks/
│   ├── 00_reproducibility_mega_notebook.ipynb
│   ├── 10_benchmark.ipynb
│   └── 20_production_annotation.ipynb
├── src/unlearning/
│   ├── config.py
│   ├── extraction.py
│   ├── prompts.py
│   ├── providers.py
│   ├── evaluation.py
│   ├── reliability.py
│   └── provenance.py
├── configs/
├── data/
├── tests/
├── runs/
├── reports/
└── archive/
```

Notebooks should explain and orchestrate. Reusable functions should eventually move into `src/unlearning/`.

In [31]:
GITIGNORE_TEMPLATE = "\n".join([
    "# Secrets",
    ".env",
    "*.key",
    "secrets/",
    "credentials/",
    "",
    "# Python",
    "__pycache__/",
    "*.py[cod]",
    ".ipynb_checkpoints/",
    ".venv/",
    "venv/",
    "",
    "# Generated and raw outputs",
    "runs/",
    "outputs/",
    "data/raw/",
    "*.jsonl.tmp",
    "*.zip",
    "*.log",
    ".DS_Store",
    "Thumbs.db",
])

REQUIREMENTS_TEMPLATE = "\n".join([
    "pandas", "numpy", "openpyxl", "scikit-learn", "statsmodels",
    "krippendorff", "python-dotenv", "tqdm", "nbformat",
    "openai", "anthropic", "google-genai", "PyMuPDF",
    "pdfplumber", "pyarrow",
])

README_TEMPLATE = (
    "# Unlearning Project\n\n"
    "Reproducible LLM-assisted coding of federal disaster-policy documents.\n\n"
    "## Reproducibility rules\n\n"
    "1. Never commit API keys.\n"
    "2. Never overwrite a historical run.\n"
    "3. Freeze codebook, dataset, prompt, model configuration, and raw responses.\n"
    "4. Report benchmark denominator and class balance.\n"
    "5. Treat model disagreement as a human-review signal.\n"
)

(PROJECT_ROOT / ".gitignore.template").write_text(GITIGNORE_TEMPLATE, encoding="utf-8")
(PROJECT_ROOT / "requirements.template.txt").write_text(REQUIREMENTS_TEMPLATE, encoding="utf-8")
(PROJECT_ROOT / "README.template.md").write_text(README_TEMPLATE, encoding="utf-8")

print("Version-control templates created in:", PROJECT_ROOT)

Version-control templates created in: /mnt/data


# 22. Final verification checklist

- [ ] All historical notebooks are present in `archive/`.
- [ ] Raw JSONL/CSV result files are preserved.
- [ ] SHA-256 hashes are populated.
- [ ] Exact dates are added when found in logs.
- [ ] The 11-row test is reported as positive recall.
- [ ] v5 contains 49 rows: 39 Yes and 10 No.
- [ ] v6 contains 42 rows: 32 Yes and 10 No.
- [ ] Duplicate and few-shot leakage rules are recorded.
- [ ] EPA, GAO, and Post-Katrina document-level results regenerate.
- [ ] Claude Sonnet’s historical 11/11 result is preserved.
- [ ] Old and new codebooks remain separate and hashed.
- [ ] Production extraction reconstructs 32 logical paragraphs.
- [ ] All 96 production predictions and 18 annotation segments are traceable.
- [ ] Raw responses are never replaced by summary tables.
- [ ] A verified bundle is created before Git refactoring.

In [32]:
SELF_AUDIT = {
    "api_calls_disabled": CONFIG.run_api_calls is False,
    "seed": CONFIG.base_seed,
    "artifact_registry_rows": len(ARTIFACT_REGISTRY),
    "experiment_registry_rows": len(EXPERIMENT_REGISTRY),
    "output_schema_sha256": sha256_text(json.dumps(OUTPUT_SCHEMA, sort_keys=True)),
    "config_sha256": sha256_text(json.dumps(asdict(CONFIG), sort_keys=True, default=str)),
    "notebook_version": CONFIG.notebook_version,
}
SELF_AUDIT

{'api_calls_disabled': True,
 'seed': 20260722,
 'artifact_registry_rows': 10,
 'experiment_registry_rows': 7,
 'output_schema_sha256': '926cea7ddf182db9f807113b2e50597518f1b1df0af984e7410997981d2b3abf',
 'config_sha256': 'c10c150c0a8a97957042074da0ffc607eaa9a9766f1592db3d95520b174e3d18',
 'notebook_version': 'mega_v1'}

## What this notebook intentionally does not do

It does not assume LangChain is automatically the next step. The immediate priority is to verify the historical record, freeze the codebook and benchmark, and identify which orchestration problems genuinely require LangChain or LangGraph.

It also does not pretend that a missing historical raw output can be reconstructed perfectly from a summary workbook. Missing evidence remains explicitly marked until the original artifact is added.